In [6]:
## W7D5 Daily Challenge:

# 1. Load and Explore the Data

#     Import the necessary libraries: pandas and sqlite3.
#     Connect to the IPL database and load the master table to understand the structure.
#     Load all the tables and print their column names to identify common columns.


import numpy as np # linear algebra
import pandas as pd
import sqlite3



In [14]:
db_path = '/content/database.sqlite'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Get all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
table_names = [table[0] for table in tables]

print("Tables in the database:")
for table_name in table_names:
    print(f"- {table_name}")

Tables in the database:
- Player
- Extra_Runs
- Batsman_Scored
- Batting_Style
- Bowling_Style
- Country
- Season
- City
- Outcome
- Win_By
- Wicket_Taken
- Venue
- Extra_Type
- Out_Type
- Toss_Decision
- Umpire
- Team
- Ball_by_Ball
- sysdiagrams
- sqlite_sequence
- Match
- Rolee
- Player_Match


In [15]:
print("\nColumn names for each table:")
column_info = {}
for table_name in table_names:
    try:
        df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 0", conn)
        columns = df.columns.tolist()
        column_info[table_name] = columns
        print(f"\nTable: {table_name}")
        print(f"Columns: {columns}")
    except Exception as e:
        print(f"Error reading table {table_name}: {e}")


Column names for each table:

Table: Player
Columns: ['Player_Id', 'Player_Name', 'DOB', 'Batting_hand', 'Bowling_skill', 'Country_Name']

Table: Extra_Runs
Columns: ['Match_Id', 'Over_Id', 'Ball_Id', 'Extra_Type_Id', 'Extra_Runs', 'Innings_No']

Table: Batsman_Scored
Columns: ['Match_Id', 'Over_Id', 'Ball_Id', 'Runs_Scored', 'Innings_No']

Table: Batting_Style
Columns: ['Batting_Id', 'Batting_hand']

Table: Bowling_Style
Columns: ['Bowling_Id', 'Bowling_skill']

Table: Country
Columns: ['Country_Id', 'Country_Name']

Table: Season
Columns: ['Season_Id', 'Man_of_the_Series', 'Orange_Cap', 'Purple_Cap', 'Season_Year']

Table: City
Columns: ['City_Id', 'City_Name', 'Country_id']

Table: Outcome
Columns: ['Outcome_Id', 'Outcome_Type']

Table: Win_By
Columns: ['Win_Id', 'Win_Type']

Table: Wicket_Taken
Columns: ['Match_Id', 'Over_Id', 'Ball_Id', 'Player_Out', 'Kind_Out', 'Fielders', 'Innings_No']

Table: Venue
Columns: ['Venue_Id', 'Venue_Name', 'City_Id']

Table: Extra_Type
Columns: ['Ex

In [21]:
# Query 1: Select All Columns from Player’s Table

# Write and execute a SQL query to select all columns from the Player_Match table.

sql='''
SELECT * FROM player_match
''';


df_sql = pd.read_sql_query(sql,con=conn)
df_sql

,Match_Id,Player_Id,Role_Id,Team_Id
0,335987,1,1,1
1,335987,2,3,1
2,335987,3,3,1
3,335987,4,3,1
4,335987,5,3,1
...,...,...,...,...
12689,981024,385,3,11
12690,981024,394,3,11
12691,981024,429,3,11
12692,981024,434,3,2


In [28]:
# Query 2: Batsman vs Runs

# Write and execute a SQL query to calculate the total runs scored by each batsman.

sql_total_runs = '''
SELECT
    P.Player_Name,
    SUM(BS.Runs_Scored) AS Total_Runs
FROM
    Ball_by_Ball BBB
JOIN
    Batsman_Scored BS ON BBB.Match_Id = BS.Match_Id
                     AND BBB.Over_Id = BS.Over_Id
                     AND BBB.Ball_Id = BS.Ball_Id
                     AND BBB.Innings_No = BS.Innings_No
JOIN
    Player P ON BBB.Striker = P.Player_Id
GROUP BY
    P.Player_Name
ORDER BY
    Total_Runs DESC;
''';

df_total_runs = pd.read_sql_query(sql_total_runs, con=conn)
display(df_total_runs.head(10))

,Player_Name,Total_Runs
0,SK Raina,4106
1,V Kohli,4105
2,RG Sharma,3874
3,G Gambhir,3634
4,CH Gayle,3447
5,RV Uthappa,3390
6,DA Warner,3373
7,MS Dhoni,3270
8,AB de Villiers,3270
9,S Dhawan,3082


In [38]:
# Query 3: Fifties and Hundreds

# Write and execute a SQL query to calculate the number of fifties and hundreds scored by each batsman.

sql = '''
WITH InningsScores AS (
    SELECT
        BBB.Match_Id,
        BBB.Innings_No,
        BBB.Striker AS Player_Id,
        SUM(BS.Runs_Scored) AS TotalRunsInInnings
    FROM
        Ball_by_Ball BBB
    JOIN
        Batsman_Scored BS ON BBB.Match_Id = BS.Match_Id
                         AND BBB.Over_Id = BS.Over_Id
                         AND BBB.Ball_Id = BS.Ball_Id
                         AND BBB.Innings_No = BS.Innings_No
    GROUP BY
        BBB.Match_Id, BBB.Innings_No, BBB.Striker
)
SELECT
    P.Player_Name,
    SUM(CASE WHEN InnS.TotalRunsInInnings >= 50 AND InnS.TotalRunsInInnings < 100 THEN 1 ELSE 0 END) AS Total_Fifties,
    SUM(CASE WHEN InnS.TotalRunsInInnings >= 100 THEN 1 ELSE 0 END) AS Total_Hundreds
FROM
    InningsScores InnS
JOIN
    Player P ON InnS.Player_Id = P.Player_Id
GROUP BY
    P.Player_Name
ORDER BY
    Total_Hundreds DESC, Total_Fifties DESC;
''';

df_fifties_hundreds = pd.read_sql_query(sql, con=conn)
display(df_fifties_hundreds.head(10))

,Player_Name,Total_Fifties,Total_Hundreds
0,CH Gayle,20,5
1,V Kohli,26,4
2,AB de Villiers,21,3
3,DA Warner,32,2
4,V Sehwag,16,2
5,SR Watson,14,2
6,M Vijay,13,2
7,AC Gilchrist,11,2
8,BB McCullum,11,2
9,RG Sharma,29,1


In [39]:
# Query 4: Best Bowling Figures

#     Write and execute a SQL query to find the best bowling figures for each bowler.


sql_best_bowling = '''
WITH BowlerStats AS (
    SELECT
        BBB.Match_Id,
        BBB.Innings_No,
        BBB.Bowler AS Player_Id,
        SUM(BS.Runs_Scored) + SUM(CASE WHEN ER.Extra_Runs IS NOT NULL THEN ER.Extra_Runs ELSE 0 END) AS Runs_Conceded,
        SUM(CASE WHEN WT.Player_Out IS NOT NULL AND WT.Kind_Out NOT IN ('run out', 'retired hurt', 'obstructing the field') THEN 1 ELSE 0 END) AS Wickets_Taken
    FROM
        Ball_by_Ball BBB
    LEFT JOIN
        Batsman_Scored BS ON BBB.Match_Id = BS.Match_Id
                             AND BBB.Over_Id = BS.Over_Id
                             AND BBB.Ball_Id = BS.Ball_Id
                             AND BBB.Innings_No = BS.Innings_No
    LEFT JOIN
        Extra_Runs ER ON BBB.Match_Id = ER.Match_Id
                         AND BBB.Over_Id = ER.Over_Id
                         AND BBB.Ball_Id = ER.Ball_Id
                         AND BBB.Innings_No = ER.Innings_No
    LEFT JOIN
        Wicket_Taken WT ON BBB.Match_Id = WT.Match_Id
                           AND BBB.Over_Id = WT.Over_Id
                           AND BBB.Ball_Id = WT.Ball_Id
                           AND BBB.Innings_No = WT.Innings_No
    GROUP BY
        BBB.Match_Id, BBB.Innings_No, BBB.Bowler
)
SELECT
    P.Player_Name,
    MAX(BS.Wickets_Taken) AS Best_Wickets,
    MIN(BS.Runs_Conceded) AS Fewest_Runs
FROM
    BowlerStats BS
JOIN
    Player P ON BS.Player_Id = P.Player_Id
GROUP BY
    P.Player_Name
ORDER BY
    Best_Wickets DESC, Fewest_Runs ASC;
''';

df_best_bowling = pd.read_sql_query(sql_best_bowling, con=conn)
display(df_best_bowling.head(10))

,Player_Name,Best_Wickets,Fewest_Runs
0,DJG Sammy,6,4
1,AD Russell,6,6
2,A Zampa,6,9
3,Sohail Tanvir,6,10
4,RA Jadeja,5,0
5,BJ Hodge,5,2
6,JD Unadkat,5,3
7,MM Patel,5,3
8,R Ashwin,5,3
9,KV Sharma,5,5


## Query 5: Comprehensive Career Metrics

Let's combine all the previous chunks into a single comprehensive query to get detailed career metrics for players. This query will include total runs, total fifties, total hundreds, best bowling wickets, and fewest runs conceded.

In [41]:
# Query 4: Best Bowling Figures

#     Write and execute a SQL query to find the best bowling figures for each bowler.

sql_career_metrics = '''
WITH PlayerTotalRuns AS (
    SELECT
        P.Player_Id,
        P.Player_Name,
        SUM(BS.Runs_Scored) AS Total_Runs
    FROM
        Ball_by_Ball BBB
    JOIN
        Batsman_Scored BS ON BBB.Match_Id = BS.Match_Id
                             AND BBB.Over_Id = BS.Over_Id
                             AND BBB.Ball_Id = BS.Ball_Id
                             AND BBB.Innings_No = BS.Innings_No
    JOIN
        Player P ON BBB.Striker = P.Player_Id
    GROUP BY
        P.Player_Id, P.Player_Name
),
InningsScores AS (
    SELECT
        BBB.Match_Id,
        BBB.Innings_No,
        BBB.Striker AS Player_Id,
        SUM(BS.Runs_Scored) AS TotalRunsInInnings
    FROM
        Ball_by_Ball BBB
    JOIN
        Batsman_Scored BS ON BBB.Match_Id = BS.Match_Id
                         AND BBB.Over_Id = BS.Over_Id
                         AND BBB.Ball_Id = BS.Ball_Id
                         AND BBB.Innings_No = BS.Innings_No
    GROUP BY
        BBB.Match_Id, BBB.Innings_No, BBB.Striker
),
PlayerBattingMilestones AS (
    SELECT
        InnS.Player_Id,
        SUM(CASE WHEN InnS.TotalRunsInInnings >= 50 AND InnS.TotalRunsInInnings < 100 THEN 1 ELSE 0 END) AS Total_Fifties,
        SUM(CASE WHEN InnS.TotalRunsInInnings >= 100 THEN 1 ELSE 0 END) AS Total_Hundreds
    FROM
        InningsScores InnS
    GROUP BY
        InnS.Player_Id
),
BowlerStats AS (
    SELECT
        BBB.Match_Id,
        BBB.Innings_No,
        BBB.Bowler AS Player_Id,
        SUM(COALESCE(BS.Runs_Scored, 0)) + SUM(COALESCE(ER.Extra_Runs, 0)) AS Runs_Conceded,
        SUM(CASE WHEN WT.Player_Out IS NOT NULL AND WT.Kind_Out NOT IN ('run out', 'retired hurt', 'obstructing the field') THEN 1 ELSE 0 END) AS Wickets_Taken
    FROM
        Ball_by_Ball BBB
    LEFT JOIN
        Batsman_Scored BS ON BBB.Match_Id = BS.Match_Id
                             AND BBB.Over_Id = BS.Over_Id
                             AND BBB.Ball_Id = BS.Ball_Id
                             AND BBB.Innings_No = BS.Innings_No
    LEFT JOIN
        Extra_Runs ER ON BBB.Match_Id = ER.Match_Id
                         AND BBB.Over_Id = ER.Over_Id
                         AND BBB.Ball_Id = ER.Ball_Id
                         AND BBB.Innings_No = ER.Innings_No
    LEFT JOIN
        Wicket_Taken WT ON BBB.Match_Id = WT.Match_Id
                           AND BBB.Over_Id = WT.Over_Id
                           AND BBB.Ball_Id = WT.Ball_Id
                           AND BBB.Innings_No = WT.Innings_No
    GROUP BY
        BBB.Match_Id, BBB.Innings_No, BBB.Bowler
),
PlayerBestBowling AS (
    SELECT
        BS.Player_Id,
        MAX(BS.Wickets_Taken) AS Best_Wickets,
        MIN(BS.Runs_Conceded) AS Fewest_Runs
    FROM
        BowlerStats BS
    GROUP BY
        BS.Player_Id
)
SELECT
    P.Player_Name,
    COALESCE(PTR.Total_Runs, 0) AS Total_Runs,
    COALESCE(PBM.Total_Fifties, 0) AS Total_Fifties,
    COALESCE(PBM.Total_Hundreds, 0) AS Total_Hundreds,
    COALESCE(PBB.Best_Wickets, 0) AS Best_Wickets,
    COALESCE(PBB.Fewest_Runs, 0) AS Fewest_Runs
FROM
    Player P
LEFT JOIN
    PlayerTotalRuns PTR ON P.Player_Id = PTR.Player_Id
LEFT JOIN
    PlayerBattingMilestones PBM ON P.Player_Id = PBM.Player_Id
LEFT JOIN
    PlayerBestBowling PBB ON P.Player_Id = PBB.Player_Id
ORDER BY
    Total_Runs DESC, Total_Hundreds DESC, Total_Fifties DESC, Best_Wickets DESC, Fewest_Runs ASC;
'''

df_career_metrics = pd.read_sql_query(sql_career_metrics, con=conn)
display(df_career_metrics.head(10))

,Player_Name,Total_Runs,Total_Fifties,Total_Hundreds,Best_Wickets,Fewest_Runs
0,SK Raina,4106,28,1,2,0
1,V Kohli,4105,26,4,2,1
2,RG Sharma,3874,29,1,4,1
3,G Gambhir,3634,31,0,0,0
4,CH Gayle,3447,20,5,3,3
5,RV Uthappa,3390,17,0,0,0
6,DA Warner,3373,32,2,0,0
7,AB de Villiers,3270,21,3,0,0
8,MS Dhoni,3270,16,0,0,0
9,S Dhawan,3082,25,0,1,4
